In [ ]:
# DUMMY NOTEBOOK — ARCHITECTURE SANITY CHECK
# No Drive saving, no JSON, no plots
# Just training + evaluation with tqdm


import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, hamming_loss, accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:

df = pd.read_csv('../../data/t2d_data.csv')
print(f"Loaded {len(df)} rows")

def clean_dia_life(x):
    if pd.isna(x):
        return np.nan
    x_str = str(x).lower().strip()
    if 'month' in x_str or x_str.endswith('m'):
        num = ''.join(filter(lambda c: c.isdigit() or c == '.', x_str))
        try:
            return float(num) / 12
        except:
            return np.nan
    else:
        try:
            return float(x_str)
        except:
            return np.nan

df['DIA LIFE'] = df['DIA LIFE'].apply(clean_dia_life)

complications = ['NEP', 'NEU', 'RET']
exclude_cols  = ['SL.NO', 'NAME'] + complications
feature_cols  = [c for c in df.columns if c not in exclude_cols]
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

df_clean = df.dropna(subset=feature_cols + complications)
print(f"After dropping missing: {len(df_clean)} rows")

X = df_clean[feature_cols].values.astype(np.float32)
y = df_clean[complications].values.astype(np.float32)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

print("\nPositive counts:")
for i, comp in enumerate(complications):
    pos = (y[:, i] == 1).sum()
    print(f"  {comp}: {pos} ({pos/len(y)*100:.1f}%)")

y_combined = y.dot(2**np.arange(y.shape[1]))
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y_combined
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42,
    stratify=y_temp.dot(2**np.arange(y_temp.shape[1]))
)

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train)}")
print(f"  Val:   {len(X_val)}")
print(f"  Test:  {len(X_test)}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

batch_size   = 32  # smaller batch for smaller network
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[1]
print(f"\nInput dimension: {input_dim}")
print("\n Preprocessing complete.")

In [ ]:

class SharedBackbone(nn.Module):
    """
    Constrained backbone.
    hidden_dim=16, proj_dim=8, n_hidden_layers configurable.
    """
    def __init__(self, input_dim, hidden_dim=16, n_hidden_layers=2, proj_dim=8):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers += [
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
            ]
            in_dim = hidden_dim
        layers += [
            nn.Linear(hidden_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        ]
        self.net        = nn.Sequential(*layers)
        self.output_dim = proj_dim

    def forward(self, x):
        return self.net(x)


class BaselineModel(nn.Module):
    """Shared backbone + independent linear head."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(backbone.output_dim, 3)

    def forward(self, x):
        return self.head(self.backbone(x))


class LinearAdditiveHead(nn.Module):
    """Linear additive interaction head."""
    def __init__(self, hidden_dim, n_labels=3, init_scale=0.01):
        super().__init__()
        self.base = nn.Linear(hidden_dim, n_labels)
        self.A    = nn.Parameter(torch.zeros(n_labels, n_labels))
        with torch.no_grad():
            self.A.data.fill_(init_scale)
            self.A.data.fill_diagonal_(0)
        mask = 1 - torch.eye(n_labels)
        self.register_buffer('mask', mask)

    def forward(self, h):
        base_logits = self.base(h)
        base_probs  = torch.sigmoid(base_logits)
        A_masked    = self.A * self.mask
        interaction = base_probs @ A_masked.T
        return base_logits + interaction


class LinearAdditiveModel(nn.Module):
    """Shared backbone + linear additive interaction head."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = LinearAdditiveHead(backbone.output_dim, n_labels=3)

    def forward(self, x):
        return self.head(self.backbone(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick param count check
for n_layers in [1, 2, 3,4,5]:
    bb    = SharedBackbone(input_dim, n_hidden_layers=n_layers)
    base  = BaselineModel(bb)
    print(f"{n_layers} hidden layers → {count_params(base):,} params  "
          f"({len(X_train)/count_params(base):.2f} samples/param)")

print("\n Model definitions ready.")

In [ ]:

def train_model(model, train_loader, val_loader, model_name,
                epochs=300, lr=0.001, patience=40):
    model     = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=15, factor=0.5
    )

    train_losses     = []
    val_aurocs       = []
    best_val_auroc   = 0
    patience_counter = 0
    best_state       = None

    pbar = tqdm(range(epochs), desc=f'{model_name}', unit='epoch',
                bar_format='{l_bar}{bar:30}{r_bar}')

    for epoch in pbar:
        #  Train 
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        #  Validate 
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                all_logits.append(model(X_batch.to(device)).cpu())
                all_labels.append(y_batch.cpu())

        all_probs  = torch.sigmoid(torch.cat(all_logits))
        all_labels = torch.cat(all_labels)
        val_auroc  = roc_auc_score(all_labels.numpy(), all_probs.numpy(), average='macro')
        val_aurocs.append(val_auroc)

        scheduler.step(val_auroc)
        lr_now = optimizer.param_groups[0]['lr']

        pbar.set_postfix({
            'loss':      f'{epoch_loss:.4f}',
            'val_auroc': f'{val_auroc:.4f}',
            'best':      f'{best_val_auroc:.4f}',
            'lr':        f'{lr_now:.6f}',
            'patience':  f'{patience_counter}/{patience}'
        })

        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                pbar.set_description(f'{model_name} [EARLY STOP ep={epoch+1}]')
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_aurocs, best_val_auroc


print(" Training function ready.")


In [ ]:

def evaluate_model(model, test_loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            all_logits.append(model(X_batch.to(device)).cpu())
            all_labels.append(y_batch)

    probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    labels = torch.cat(all_labels).numpy()
    preds  = (probs > 0.5).astype(int)

    per_label_auroc = {
        comp: roc_auc_score(labels[:, i], probs[:, i])
        for i, comp in enumerate(complications)
    }

    return {
        'auroc_macro':      roc_auc_score(labels, probs, average='macro'),
        'per_label_auroc':  per_label_auroc,
        'f1_macro':         f1_score(labels, preds, average='macro'),
        'hamming_loss':     hamming_loss(labels, preds),
        'subset_accuracy':  accuracy_score(labels, preds),
    }


print(" Evaluation function ready.")

In [ ]:

N_LAYERS_TO_TEST = [1, 2, 3,4,5]
SEED             = 42
results          = {}

print("\n" + "="*65)
print("DUMMY RUN — TESTING DEPTHS: 1, 2, 3 hidden layers")
print(f"Architecture: hidden_dim=16, proj_dim=8")
print(f"Dataset: {len(X_train)} train samples, {input_dim} features")
print("="*65)

for n_layers in N_LAYERS_TO_TEST:
    print(f"\n{'='*60}")
    print(f"DEPTH: {n_layers} hidden layer(s)")
    print(f"{'='*60}")

    set_seed(SEED)
    backbone   = SharedBackbone(input_dim, n_hidden_layers=n_layers)
    init_state = {k: v.cpu().clone() for k, v in backbone.state_dict().items()}
    n_params   = count_params(BaselineModel(backbone))
    print(f"Total params: {n_params:,}  |  "
          f"Samples/param: {len(X_train)/n_params:.2f}")

    #  Baseline 
    set_seed(SEED)
    baseline_model = BaselineModel(backbone)
    baseline_model, _, _, baseline_best_val = train_model(
        baseline_model, train_loader, val_loader,
        f'Baseline {n_layers}L'
    )
    baseline_results = evaluate_model(baseline_model, test_loader)

    #  Linear Additive 
    backbone_lin = SharedBackbone(input_dim, n_hidden_layers=n_layers)
    backbone_lin.load_state_dict(init_state)

    set_seed(SEED)
    linear_model = LinearAdditiveModel(backbone_lin)
    linear_model, _, _, linear_best_val = train_model(
        linear_model, train_loader, val_loader,
        f'LinAdd  {n_layers}L'
    )
    linear_results = evaluate_model(linear_model, test_loader)

    gap = linear_results['auroc_macro'] - baseline_results['auroc_macro']
    results[n_layers] = {
        'baseline': baseline_results,
        'linear':   linear_results,
        'gap':      gap
    }

    print(f"\n  Baseline AUROC:        {baseline_results['auroc_macro']:.4f}")
    print(f"  Linear Additive AUROC: {linear_results['auroc_macro']:.4f}")
    print(f"  Gap:                   {gap:+.4f}")
    print(f"  Per-label (Base):  "
          f"NEP={baseline_results['per_label_auroc']['NEP']:.4f}  "
          f"NEU={baseline_results['per_label_auroc']['NEU']:.4f}  "
          f"RET={baseline_results['per_label_auroc']['RET']:.4f}")
    print(f"  Per-label (LinAdd):"
          f"NEP={linear_results['per_label_auroc']['NEP']:.4f}  "
          f"NEU={linear_results['per_label_auroc']['NEU']:.4f}  "
          f"RET={linear_results['per_label_auroc']['RET']:.4f}")

In [ ]:

print("\n" + "="*65)
print("DUMMY RUN SUMMARY")
print("="*65)
print(f"\n{'Depth':<10} {'Base AUROC':<14} {'LinAdd AUROC':<16} {'Gap':<12} {'Winner'}")
print("-"*60)
for n_layers in N_LAYERS_TO_TEST:
    r      = results[n_layers]
    b_auc  = r['baseline']['auroc_macro']
    l_auc  = r['linear']['auroc_macro']
    gap    = r['gap']
    winner = 'Linear' if gap > 0 else 'Baseline'
    print(f"{n_layers} layer(s)  {b_auc:<14.4f} {l_auc:<16.4f} {gap:<+12.4f} {winner}")

print("\nWhat to look for:")
print("  1. Are AUROC values meaningfully below 0.97? (task not too easy)")
print("  2. Is there a consistent gap in favour of Linear Additive?")
print("  3. Which depth gives the best baseline without overfitting?")